In [ ]:
import os
import json 
import scipy.stats as stats
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from natural_quantization.preprocess import prepare_data, read_weights, hot_encode
from qiskit_ibm_runtime.fake_provider import FakeManilaV2, FakeKyiv

# TODO: enforce one class"ical data type
# TODO: make sure we select qubits with furthest connectivity
# TODO : pytorch mnist dataset 

BASE_DIR = "/Users/dlakhdar/physics/repos/natural-quantization/data/quantum_nn_rotation_angles"
MNIST_DATA_PATH = "/Users/dlakhdar/physics/repos/natural-quantization/data/quantum_nn_rotation_angles/mnist_data/mnist_data_reduced.json"
DATA_DUMP_PATH= "/Users/dlakhdar/physics/repos/natural-quantization/data/quantum_nn_rotation_angles/experiment_data"

def calculate_validation_rate(predicted_y: list[np.ndarray],
                              y: list[np.ndarray]) -> float | int:
    """
    Calculate the validation rate (accuracy) for predicted and actual labels.

    Parameters:
    - predicted_y: array-like, predicted probabilities or logits (e.g., from a neural network).
    - y: array-like, one-hot encoded true labels.

    Returns:
    - float, the accuracy rate as the proportion of correctly predicted samples.
    """
    predicted_indices = np.array(list(map(np.argmax, predicted_y)))
    true_indices = np.array(list(map(np.argmax, y)))
    accuracy = np.mean(predicted_indices == true_indices)
    return accuracy

def mode_of_argmaxes(list_of_lists: list[list[list[float]]]) -> list[int]:
    """
    For each sub‐list `l` in `list_of_lists`:
      1. Compute argmax of each inner list `l'`.
      2. Take the statistical mode of those argmaxes.
    Returns a list of one mode‐index per `l`.
    """
    modes = []
    for l in list_of_lists:
        # 1) argmax of each l'
        argmaxes = [int(np.argmax(l_prime)) for l_prime in l]
        # 2) most common
        mode_idx = int(stats.mode(argmaxes)[0])
        modes.append(mode_idx)
    return modes


def one_hot(digit: int, length: int = 10) -> np.ndarray:
    """
    Return a one‑hot vector of given length with a 1 at position `digit`.
    
    Parameters
    ----------
    digit : int
        An integer between 0 and length‑1 inclusive.
    length : int
        Size of the output vector (default 10).
        
    Returns
    -------
    np.ndarray
        One‑hot encoded vector of shape (length,).
    """
    if not (0 <= digit < length):
        raise ValueError(f"digit must be in [0, {length-1}]")
    vec = np.zeros(length, dtype=int)
    vec[digit] = 1
    return vec


def htanh(x: np.ndarray[:], a: (int | float)) -> np.ndarray[:]:
    """
    Compute a hard tanh activation on the input x.

    This function applies a piecewise transformation to x:
      - When |x| ≤ a, it returns x/a.
      - When |x| > a, it returns 1 if x is positive, and -1 otherwise.
      - For a = 0, the function directly returns the sign of x.

    Parameters:
      x : array-like or scalar
          The input value(s) to be transformed.
      a : float
          The threshold parameter (with |a| ≤ 1) that defines the linear region.

    Returns:
      Transformed value(s) following the hard tanh definition applied element-wise.

    Raises:
      AssertionError: If the absolute value of a is greater than 1.
    """

    assert abs(a) <= 1

    if a == 0:
        # purely sign‐based when a==0
        return np.sign(x)
    # scale into [−1,1] and clip
    return np.clip(x / a, -1, 1)

class QuantumNeuralNetwork:

    """
    A hybrid classical–quantum feedforward neural network built on Qiskit Runtime.

    This class implements a multi-layer network where the first and last layers
    are classical linear transforms and intermediate hidden layers are executed
    as parameterized single-qubit quantum circuits. Each hidden layer’s pre‑activation
    vector is converted into rotation angles, the qubits are measured, and the
    resulting bitstring is used as input to the next layer. The final layer applies
    a classical softmax to produce output probabilities.

    Attributes
    ----------
    layer_sizes : list of int
        Sizes of each layer, including input, hidden, and output layers.
    layer_n : int
        Total number of layers.
    hidden_layer_n : int
        Number of hidden (quantum) layers.
    b : list of np.ndarray
        Bias vectors for each layer (starting from the first hidden layer).
    W : list of np.ndarray
        Weight matrices for each layer transition.
    activation_f : callable
        A function mapping (pre‑activation, quantumness) → activation, e.g. `htanh`.
    service : QiskitRuntimeService or None
        Initialized IBM runtime service after calling `establish_communication_with_ibm`.
    backend : Qiskit backend or None
        Selected device or simulator for quantum circuit execution.
    comms : pass manager or None
        Communication pass manager for circuit compilation.
    """

    def __init__(
        self, layer_sizes: list, activation_f: callable = htanh
    ) -> "QuantumNeuralNetwork":
        self.layer_sizes = layer_sizes
        self.layer_n = len(layer_sizes)
        self.hidden_layer_n = len(layer_sizes) - 2
        self.b = [
            np.random.randn(layer_sizes[i]) for i in range(1, self.hidden_layer_n + 2)
        ]
        self.W = [
            np.random.randn(layer_sizes[i], layer_sizes[i - 1])
            * np.sqrt(1 / layer_sizes[i - 1])
            for i in range(1, self.hidden_layer_n + 2)
        ]
        self.bias = self.b
        self.weights = self.W
        self.activation_f = activation_f
        self.service = None
        self.backend = None
        self.comms = None

    def establish_communication_with_ibm(
        self, simulation_mode=True, simulator=FakeKyiv, operational=True, optimization_level=3
    ) -> None:
        """
        Initialize Qiskit runtime service, select a backend, and configure the communications
        pass manager.

        This method logs into the IBM Qiskit runtime service, chooses the least-busy backend
        supporting sessions and measurements (either a simulator or a real device), and
        generates a preset pass manager with the specified optimization level for compiling
        circuits.

        Parameters
        ----------
        simulation_mode : bool, optional
            Whether to prefer a simulator backend (True) or a real device (False). Default is
            True.
        operational : bool, optional
            Whether to restrict selection to operational (online) backends (True) or include
            offline ones (False). Default is True.
        optimization_level : int, optional
            The optimization level (0–3) for the communications pass manager. Higher values apply
              more aggressive optimizations. Default is 3.

        Raises
        ------
        ConnectionError
            If initializing QiskitRuntimeService or selecting the backend fails.
        RuntimeError
            If generating the communications pass manager fails.
        """

        try:
            # Initialize the runtime service (assumes you're logged in)
            self.service = QiskitRuntimeService()
        except Exception as e:
            raise ConnectionError("Failed to initialize QiskitRuntimeService.") from e

        try:
            if simulation_mode is True :
                fakekyiv = FakeKyiv()
                self.backend = fakekyiv
            # Choose a backend with session and measurement support
            else: 
                self.backend = self.service.least_busy(simulator=simulation_mode, operational=operational
                )
        except Exception as e:
            raise ConnectionError("Failed to select a backend.") from e

        try:
            # Generate the communication pass manager
            self.comms = generate_preset_pass_manager(backend=self.backend, optimization_level=optimization_level)
        except Exception as e:
            raise RuntimeError(
                "Failed to generate the communications pass manager."
            ) from e

    def feedforward(
        self: "QuantumNeuralNetwork", input: np.ndarray[:], quantumness=0.5, shots: int=1
    ) -> np.ndarray[:]:
        """
        Perform a hybrid classical–quantum feedforward pass through the network.

        This method processes the input vector through a classical linear layer,
        then for each hidden layer builds and runs a parameterized quantum circuit
        whose rotation angles are determined by the difference between the pre‑activation
        zs and the activations computed by `self.activation_f`. The measurement outcomes
        of each quantum circuit become the inputs to the next layer. Finally, it
        applies a softmax transformation on the last linear readout layer to produce
        output probabilities.

        Parameters
        ----------
        self : QuantumNeuralNetwork
            The network instance, which must have attributes `weights`, `activation_f`,
            `comms`, and `backend` configured.
        input : np.ndarray, shape (n_input,)
            The input feature vector for the network.
        quantumness : float, optional
            A value between 0 and 1 controlling the interpolation between purely
            classical (0) and fully quantum (1) hidden‑layer activations. Default is 0.5.

        Returns
        -------
        np.ndarray, shape (n_output,)
            The output probability vector from the final softmax layer.
        """
        sampler = Sampler(mode=self.backend)
        zs = input
        for i in range(0, len(self.weights) - 1):
            # print(f"layer {i}")
            θ = np.pi / 2 * (1 - self.activation_f(self.weights[i] @ zs, quantumness))
            # print("thetas:",θ)
            n_qbits = self.weights[i].shape[0]
            qr = QuantumRegister(n_qbits, name="qr")
            cr = ClassicalRegister(n_qbits, name="cr")
            qc = QuantumCircuit(qr, cr)
            for j, ϕ in enumerate(θ):
                qc.ry(theta=ϕ, qubit=j)
                qc.measure(qubit=j, cbit=j)
            isa_circuit = self.comms.run(qc)
            job = sampler.run([isa_circuit], shots=shots)
            result = job.result()
            counts = result[0].data.cr.get_counts()
            # print("counts:", counts)
            # takes bit string e.g. '10010' and makes array [1,0,0,1,0]
            zs = np.fromiter((int(b) for b in next(iter(counts))), dtype=np.float16)
            zs = zs[::-1]
            # print("bit-string -> array:", zs)
            # set 0 to 1, 1 to -1 
            zs = np.where(zs == 0, 1, -1)
            # print("activations:", zs)

        # apply softmax to final layer

        # print("final pre-activation:", zs)
        output = np.exp(self.weights[-1] @ zs) / sum(np.exp(self.weights[-1] @ zs))
        # print("output:",output)
        return output
        

    def simulated_feedforward(
        self: "QuantumNeuralNetwork", input: np.ndarray[:], quantumness=0.5
    ) -> np.ndarray[:]:
        """
        Perform a hybrid classical–quantum feedforward pass through the network.

        This method processes the input vector through a classical linear layer,
        then for each hidden layer builds and runs a parameterized quantum circuit
        whose rotation angles are determined by the difference between the pre‑activation
        zs and the activations computed by `self.activation_f`. The measurement outcomes
        of each quantum circuit become the inputs to the next layer. Finally, it
        applies a softmax transformation on the last linear readout layer to produce
        output probabilities.

        Parameters
        ----------
        self : QuantumNeuralNetwork
            The network instance, which must have attributes `weights`, `activation_f`,
            `comms`, and `backend` configured.
        input : np.ndarray, shape (n_input,)
            The input feature vector for the network.
        quantumness : float, optional
            A value between 0 and 1 controlling the interpolation between purely
            classical (0) and fully quantum (1) hidden‑layer activations. Default is 0.5.

        Returns
        -------
        np.ndarray, shape (n_output,)
            The output probability vector from the final softmax layer.
        """
        zs = input
        for i in range(0, len(self.weights) - 1):
            θ = np.pi / 2 * (1 - self.activation_f(self.weights[i] @ zs, quantumness))
            ϵ = np.random.rand(len(θ))
            zs = ((ϵ<(1+np.cos(θ))/2)-.5)*2

        # apply softmax to final layer
        output = np.exp(self.weights[-1] @ zs) / sum(np.exp(self.weights[-1] @ zs))
        return output

    def predict(self, Xs, quantumness=0.5, n_samples=10) -> list[np.ndarray[:]]:
        """
        Generate multiple stochastic predictions for each input using the hybrid
        quantum–classical network.

        Parameters
        ----------
        Xs : Iterable[np.ndarray]
            A collection of input vectors to predict on. Each element should be a 1D numpy array
            matching the network's input dimension.
        quantumness : float, optional
            Interpolation parameter between purely classical (0.0) and fully quantum (1.0
            hidden-layer
            activations. Defaults to 0.5.
        n_samples : int, optional
            Number of stochastic forward passes to perform per input. Defaults to 10.

        Returns
        -------
        list[np.ndarray]
            A flat list of output probability vectors from all runs, with length equal to
            `len(Xs) * n_samples`. Each element is the softmax output of one forward pass.
        """

        output = []
        for x in Xs:
            per_instance = [ ]
            for i in range(n_samples):
                tmp = self.feedforward(x, quantumness=quantumness)
                # print(f"shot #{i}", np.argmax(tmp))
                per_instance.append(np.argmax(tmp))
            output.append(per_instance)
        return output
    
    def simulated_predict(self, Xs, quantumness=0.5, n_samples=10) -> list[np.ndarray[:]]:
        """
        Generate multiple stochastic predictions for each input using the hybrid
        quantum–classical network.

        Parameters
        ----------
        Xs : Iterable[np.ndarray]
            A collection of input vectors to predict on. Each element should be a 1D numpy array
            matching the network's input dimension.
        quantumness : float, optional
            Interpolation parameter between purely classical (0.0) and fully quantum (1.0
            hidden-layer
            activations. Defaults to 0.5.
        n_samples : int, optional
            Number of stochastic forward passes to perform per input. Defaults to 10.

        Returns
        -------
        list[np.ndarray]
            A flat list of output probability vectors from all runs, with length equal to
            `len(Xs) * n_samples`. Each element is the softmax output of one forward pass.
        """

        output = []
        for x in Xs:
            per_instance = [ ]
            for i in range(n_samples):
                tmp = self.simulated_feedforward(x, quantumness=quantumness)
                # print(f"shot #{i}", np.argmax(tmp))
                per_instance.append(np.argmax(tmp))
            output.append(per_instance)
        return output
    

def run_experiment(inputs: list[np.ndarray[:]],
                   data_directory: str=BASE_DIR, 
                   a_values: list[float] = [0.0, 0.1, 0.2, 0.5, 1.0],
                   layer_widths: list[int] = [16],
                   input_n: int = 784,
                   output_n: int = 10,
                   simulation_mode: bool=True,
                   n_samples: int =10) -> list[tuple[int,int,list[list]]]: 

    """

    Return: list[tuple[int,int,list[list]]]

    This is a list containing setup data. Outer list is collection of setups. The tuple contains 
    setup a, setup width, and the data from that setup. The data list is for collection of images. 
    Each image list has n_samples. 
    
    """

    setup = []
    for a in a_values:
        for width in layer_widths:
            fname = f"mnist_a{a}_lr-2_shots20_width{width}.json"
            path  = os.path.join(data_directory, fname)
            setup.append((path, a, width))

    setup_results = []

    for file_path, a, width in setup:

        print(f"a: {a} , l: {width}")
        # initialize network
        qnn = QuantumNeuralNetwork(
            layer_sizes=[input_n, width, width, width, output_n],
            activation_f=htanh
        )
        # load and assign weights
        weights = read_weights(file_path)
        qnn.weights = [np.array(w) for w in weights]

        # ensure IBM connection
        qnn.establish_communication_with_ibm(
            simulation_mode=simulation_mode,
            operational=True
        )

        # run predictions
        results = qnn.predict(inputs, quantumness=a, n_samples=n_samples)
        setup_results.append((a,width,results))

    return setup_results

In [ ]:
with open(MNIST_DATA_PATH) as f :
    data = json.load(f)

test_set =  data["test_set"]
x_test = [np.array(d[0]) for d in test_set]
y_test = [one_hot(d[1]) for d in test_set]
indices = [0,1,2,3,4,7,11,15,18,30]
# index 0 -> 7, index, index 1 ->2, index 2 ->1 , index 3 -> 0 , index 4 -> 4 , index 7 -> 9
# index 11 -> 6, index 15 -> 5, index 18 -> 8
x_test_sample = [x_test[i] for i in indices]
y_test_sample = [y_test[i] for i in indices]


In [ ]:
mid = len(x_test_sample) // 2  # → 5 for length-10
results = run_experiment(x_test_sample[mid:],simulation_mode=False, n_samples=10)

In [25]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import  Sampler  # or whatever runtime primitives you use
from qiskit.transpiler import generate_preset_pass_manager
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister


# ... inside your class/method, when NOT in simulation mode:

try:
    service = QiskitRuntimeService()
except Exception as e:
    raise ConnectionError("Failed to initialize QiskitRuntimeService.") from e

# 1) Specify the desired backend name:
desired_backend_name = "ibm_kyiv"  # or the exact name of the backend in your account

# 2) (Optional) Check it’s available and operational:
all_backends =service.backends(
    # simulator=False ensures we look only at real devices
    simulator=False,
    # You can also filter for e.g. minimum qubits, open for enrollment, etc.
)
available_names = [b.name for b in all_backends]
print("Available real-device backends:", available_names)

if desired_backend_name in available_names:
    print(f"Backend '{desired_backend_name}' is available.")
# 3) Get the backend instance:


backend = service.backend(desired_backend_name)

# Now you can create runtime options or pass self.backend when running jobs:
# e.g., for a sampler:

# comms = generate_preset_pass_manager(
#                 backend=backend, optimization_level=0
#             )


# sampler = Sampler(mode=backend)


# n_qbits = 2
# n_blocks = 1  # number of blocks in the circuit, e.g. for a single layer
#             # print(f"total qubits: {n_qbits*n_blocks}")
# qr = QuantumRegister(n_qbits * n_blocks, name="qr")
# cr = ClassicalRegister(n_qbits * n_blocks, name="cr")
# qc = QuantumCircuit(qr, cr)

# for k, ϕ in enumerate([.5,1.0]):
#     qc.ry(theta=ϕ, qubit=k)
#     qc.measure(qubit=k, cbit=k)


# isa_circuit = comms.run(qc)
# result = sampler.run([isa_circuit], shots=1).result()


# counts = result[0].data.cr.get_counts()
#             # print("counts:", counts)
#             # takes bit string e.g. '10010' and makes array [1,0,0,1,0]


/var/folders/04/2cqfhcv133s3gbkf3b35tnkr0000gn/T/ipykernel_6412/2182613498.py:10: DeprecationWarning: The "ibm_quantum" channel option is deprecated and will be sunset on 1 July. After this date, "ibm_cloud" and "local" will be the only valid channels. For information on migrating to the new IBM Quantum Platform on the "ibm_cloud" channel, review the migration guide https://quantum.cloud.ibm.com/docs/migration-guides/classic-iqp-to-cloud-iqp .
  service = QiskitRuntimeService()


Available real-device backends: ['ibm_brisbane', 'ibm_sherbrooke']


QiskitBackendNotFoundError: 'No backend matches the criteria.'

In [20]:
import numpy as np
zs = np.fromiter((int(b) for b in next(iter(counts))),
                             dtype=np.float16)
zs = zs[::-1]

In [ ]:
zs

array([0., 0.], dtype=float16)